In [1]:
from pathlib import Path
from torch.utils.data import DataLoader
from rf_learning_dataset import RFLearningDataset

EXP = {
    "experiment_name": "tiny_nonpoint_full_32",
    "project_root": "/home/liujia/RF_Image",

    "include_categories": ["carotid", "muscle", "phantom"],

    "batch_size": 16,
    "num_epochs": 100,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "normalize": True,
    "abs_weight": 0.1,

    "model_name": "tiny",
    "hidden": 64,
    "seed": 20260522,
}

PROJECT_ROOT = Path(EXP["project_root"])
DATA_ROOT = PROJECT_ROOT / "Data"

CKPT_DIR = PROJECT_ROOT / "checkpoint" / EXP["experiment_name"]
METRIC_DIR = PROJECT_ROOT / "test_metrics" / EXP["experiment_name"]
VIS_DIR = PROJECT_ROOT / "vis_best_model" / EXP["experiment_name"]

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT :", DATA_ROOT)
print("CKPT_DIR  :", CKPT_DIR)
print("METRIC_DIR:", METRIC_DIR)
print("VIS_DIR   :", VIS_DIR)

train_set = RFLearningDataset(
    root_dir=DATA_ROOT / "train",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

val_set = RFLearningDataset(
    root_dir=DATA_ROOT / "val",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

test_set = RFLearningDataset(
    root_dir=DATA_ROOT / "test",
    sample_group="/sample_000001",
    normalize=EXP["normalize"],
    include_categories=EXP["include_categories"],
)

train_loader = DataLoader(
    train_set,
    batch_size=EXP["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_set,
    batch_size=EXP["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

test_loader = DataLoader(
    test_set,
    batch_size=EXP["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

print("Train:", len(train_set))
print("Val  :", len(val_set))
print("Test :", len(test_set))

batch = next(iter(train_loader))
print("input   :", batch["input"].shape)
print("label   :", batch["label"].shape)
print("baseline:", batch["baseline"].shape)
print("category:", batch["category"])




DATA_ROOT : /home/liujia/RF_Image/Data
CKPT_DIR  : /home/liujia/RF_Image/checkpoint/tiny_nonpoint_full_32
METRIC_DIR: /home/liujia/RF_Image/test_metrics/tiny_nonpoint_full_32
VIS_DIR   : /home/liujia/RF_Image/vis_best_model/tiny_nonpoint_full_32
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/Data/train
  samples    : 1050
  normalize  : True
  include    : {'muscle', 'carotid', 'phantom'}
  exclude    : set()
  carotid   : 350
  muscle    : 350
  phantom   : 350
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/Data/val
  samples    : 225
  normalize  : True
  include    : {'muscle', 'carotid', 'phantom'}
  exclude    : set()
  carotid   : 75
  muscle    : 75
  phantom   : 75
RFLearningDataset
  root_dir   : /home/liujia/RF_Image/Data/test
  samples    : 225
  normalize  : True
  include    : {'muscle', 'carotid', 'phantom'}
  exclude    : set()
  carotid   : 75
  muscle    : 75
  phantom   : 75
Train: 1050
Val  : 225
Test : 225
input   : torch.Size([16, 1536, 32, 16, 16])

In [2]:
import time
import torch

def benchmark_loader_and_compute(model, loader, device, n_batches=20):
    model.eval()

    data_times = []
    compute_times = []

    it = iter(loader)

    for i in range(n_batches):
        t0 = time.time()
        batch = next(it)
        t1 = time.time()

        x = batch["input"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        b = batch["baseline"].to(device, non_blocking=True)

        torch.cuda.synchronize()
        t2 = time.time()

        with torch.no_grad():
            pred = model(x, b)
            loss = torch.mean(torch.abs(pred - y))

        torch.cuda.synchronize()
        t3 = time.time()

        data_times.append(t1 - t0)
        compute_times.append(t3 - t2)

    print(f"Avg data loading time: {sum(data_times)/len(data_times):.4f} s")
    print(f"Avg GPU compute time : {sum(compute_times)/len(compute_times):.4f} s")
    print(f"Data / compute ratio : {(sum(data_times)/len(data_times)) / (sum(compute_times)/len(compute_times) + 1e-12):.2f}")

In [10]:
benchmark_loader_and_compute(model, train_loader, device, n_batches=20)

Avg data loading time: 2.3147 s
Avg GPU compute time : 0.0238 s
Data / compute ratio : 97.26


In [3]:
from rf_models import build_model
from rf_train_utils import (
    seed_everything,
    get_device,
    count_trainable_parameters,
    train_model_jupyter,
    plot_training_curve,
)

seed_everything(EXP["seed"])

device = get_device()
print("Device:", device)

model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

print(f"Trainable parameters: {count_trainable_parameters(model) / 1e6:.3f} M")

history = train_model_jupyter(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    ckpt_dir=CKPT_DIR,
    experiment_name=EXP["experiment_name"],
    num_epochs=EXP["num_epochs"],
    lr=EXP["lr"],
    weight_decay=EXP["weight_decay"],
    abs_weight=EXP["abs_weight"],
    print_every=5,
    seed=EXP["seed"],
    config=EXP,
)

Device: cuda
Trainable parameters: 0.265 M

Initial validation:
val_L1=3.999772e-01 | baseline_L1=3.999772e-01 | improvement=0.00%
Epoch 0001 | train_L1=2.617520e-01 | train_base=4.085750e-01 | train_impr= 35.94% | val_L1=2.125004e-01 | val_base=3.999772e-01 | val_impr= 46.87% | lr=1.00e-03
Epoch 0005 | train_L1=2.026131e-01 | train_base=4.085081e-01 | train_impr= 50.40% | val_L1=2.041017e-01 | val_base=3.999772e-01 | val_impr= 48.97% | lr=9.94e-04
Epoch 0010 | train_L1=1.834453e-01 | train_base=4.084300e-01 | train_impr= 55.09% | val_L1=2.042237e-01 | val_base=3.999772e-01 | val_impr= 48.94% | lr=9.76e-04
Epoch 0015 | train_L1=1.778050e-01 | train_base=4.085744e-01 | train_impr= 56.48% | val_L1=2.042544e-01 | val_base=3.999772e-01 | val_impr= 48.93% | lr=9.46e-04
Epoch 0020 | train_L1=1.747543e-01 | train_base=4.083972e-01 | train_impr= 57.21% | val_L1=2.049594e-01 | val_base=3.999772e-01 | val_impr= 48.76% | lr=9.05e-04
Epoch 0025 | train_L1=1.726246e-01 | train_base=4.086596e-01 | t

KeyboardInterrupt: 

In [12]:
import torch
import pandas as pd

from rf_models import build_model
from rf_eval_utils import (
    evaluate_full_test_set,
    summarize_test_metrics,
    save_test_summaries,
    find_worse_samples,
)

from rf_visualization import (
    visualize_model_samples,
    select_indices_from_metrics_df,
)

# ============================================================
# Load best model
# ============================================================

best_model = build_model(
    EXP["model_name"],
    in_channels=1536,
    hidden=EXP["hidden"],
    out_channels=2,
).to(device)

ckpt_path = CKPT_DIR / "best_model.pth"
ckpt = torch.load(ckpt_path, map_location=device)

best_model.load_state_dict(ckpt["model"])
best_model.eval()

print("Loaded best model")
print("  epoch       :", ckpt["epoch"])
print("  best val L1 :", ckpt["best_val_l1"])
print("  ckpt path   :", ckpt_path)

# ============================================================
# Full test evaluation
# ============================================================

df_test = evaluate_full_test_set(
    model=best_model,
    dataset=test_set,
    device=device,
    batch_size=EXP["batch_size"],
    save_csv_path=METRIC_DIR / "test_per_sample_metrics.csv",
)

overall, cat_summary = summarize_test_metrics(df_test)

save_test_summaries(
    df=df_test,
    metric_dir=METRIC_DIR,
    overall=overall,
    cat_summary=cat_summary,
    prefix="test",
)

# ============================================================
# Worse samples
# ============================================================

worse_complex = find_worse_samples(
    df_test,
    metric="complex",
    top_k=20,
)

worse_abs = find_worse_samples(
    df_test,
    metric="abs",
    top_k=20,
)

# 也保存一下，方便后面查
worse_complex.to_csv(METRIC_DIR / "test_worse_complex_top20.csv", index=False)
worse_abs.to_csv(METRIC_DIR / "test_worse_abs_top20.csv", index=False)

# ============================================================
# Visualization: random/category samples
# ============================================================

vis_random = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "random_by_category",
    indices=None,
    n_per_category=3,
    categories=EXP["include_categories"],
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="random",
)

# ============================================================
# Visualization: worst complex samples
# ============================================================

worst_indices = select_indices_from_metrics_df(
    dataset=test_set,
    df=df_test,
    top_k=9,
    metric="complex_improvement",
    ascending=True,
)

vis_worst = visualize_model_samples(
    model=best_model,
    dataset=test_set,
    device=device,
    save_dir=VIS_DIR / "worst_complex",
    indices=worst_indices,
    view="xz",
    slice_index=None,
    db_min=-60,
    show=False,
    prefix="worst_complex",
)

print("\nEvaluation package finished.")
print("Metric dir:", METRIC_DIR)
print("Vis dir   :", VIS_DIR)
print("Random visualizations:", len(vis_random))
print("Worst visualizations :", len(vis_worst))

Loaded best model
  epoch       : 7
  best val L1 : 0.2015533834695816
  ckpt path   : /home/liujia/RF_Image/checkpoint/tiny_baseline_cond_nonpoint_full_32/best_model.pth
Saved per-sample metrics to: /home/liujia/RF_Image/test_metrics/tiny_baseline_cond_nonpoint_full_32/test_per_sample_metrics.csv

================ Overall test summary ================
Samples: 225

[Complex L1]
pred mean       : 3.6653e+03
baseline mean   : 6.5679e+03
mean improvement: 47.07%
better rate     : 99.56%

[Envelope abs L1]
pred mean       : 3.5075e+03
baseline mean   : 6.7495e+03
mean improvement: 48.98%
better rate     : 98.22%

================ Per-category summary ================


,n,complex_pred_mean,complex_base_mean,complex_improvement_mean,complex_better_rate,abs_pred_mean,abs_base_mean,abs_improvement_mean,abs_better_rate
category,,,,,,,,,
carotid,75,5724.107174,9206.067879,0.422285,0.986667,5580.546799,9166.495579,0.423718,0.946667
muscle,75,4042.951178,7999.789443,0.487613,1.000000,3727.000018,8546.016800,0.555378,1.000000
phantom,75,1228.873328,2497.784227,0.502281,1.000000,1214.912189,2535.845108,0.490383,1.000000


Saved test summaries to: /home/liujia/RF_Image/test_metrics/tiny_baseline_cond_nonpoint_full_32
complex worse samples: 1


,path,category,pred_complex_l1,base_complex_l1,complex_improvement,complex_better,pred_abs_l1,base_abs_l1,abs_improvement,abs_better
26,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,18628.361328,12293.327148,-0.515323,False,17954.160156,11995.667969,-0.49672,False


abs worse samples: 4


,path,category,pred_complex_l1,base_complex_l1,complex_improvement,complex_better,pred_abs_l1,base_abs_l1,abs_improvement,abs_better
26,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,18628.361328,12293.327148,-0.515323,False,17954.160156,11995.667969,-0.496720,False
50,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,17126.324219,18228.386719,0.060459,True,22395.097656,15642.389648,-0.431693,False
56,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,19742.462891,22921.765625,0.138702,True,23285.632812,21409.347656,-0.087639,False
12,/home/liujia/RF_Image/Data/test/carotid/RF0004...,carotid,5173.667969,5965.074707,0.132673,True,5450.670410,5143.250000,-0.059772,False


Selected indices: [0, 1, 2, 75, 76, 77, 150, 151, 152]
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full_32/random_by_category/001_random_carotid_RF000486_carotid_test_patch001.png
  complex L1 pred/base: 8.8592e+03 / 1.6228e+04
  abs     L1 pred/base: 1.0339e+04 / 1.5871e+04
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full_32/random_by_category/002_random_carotid_RF000486_carotid_test_patch002.png
  complex L1 pred/base: 8.4040e+03 / 1.8881e+04
  abs     L1 pred/base: 6.4186e+03 / 2.2192e+04
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full_32/random_by_category/003_random_carotid_RF000486_carotid_test_patch003.png
  complex L1 pred/base: 6.5694e+03 / 9.7930e+03
  abs     L1 pred/base: 8.2922e+03 / 8.3464e+03
Saved: /home/liujia/RF_Image/vis_best_model/tiny_baseline_cond_nonpoint_full_32/random_by_category/004_random_muscle_RF000386_muscle_test_patch001.png
  complex L1 pred/base: 3.0471e+03 / 6.7757e+03